# Лабораторная работа №4 — случайный лес

## 1. Выбор начальных условий

- **Spambase:** по 57 признакам письма определяем, спам ли оно. Основная метрика F1-macro одинаково учитывает обе метки; accuracy показывает долю верных ответов.
- **SML2010:** по 18 текущим признакам прогнозируем температуру в столовой через 15 минут. MAE показывает среднюю абсолютную ошибку в °C, а RMSE сильнее штрафует крупные ошибки.

Используем те же стратифицированное и хронологическое разбиения, что в [ЛР1](01_knn.ipynb). Подготовка данных описана в [протоколе оценки](../docs/data-protocol.md). Загружаем обе пары выборок и выводим их размеры.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from data_setup import DINING_TEMPERATURE, RANDOM_STATE, load_sml2010, load_spambase, sml2010_cv, spambase_cv
from notebooks.tree import DecisionTreeNumPy

Xc_train, Xc_test, yc_train, yc_test = load_spambase()
Xr_train, Xr_test, yr_train, yr_test = load_sml2010()
print(f"Spambase: {Xc_train.shape} / {Xc_test.shape}")
print(f"SML2010:  {Xr_train.shape} / {Xr_test.shape}")

Spambase: (3363, 57) / (841, 57)
SML2010:  (2763, 18) / (1370, 18)


## 2. Создание бейзлайна и оценка качества

Обучаем леса sklearn из 25 деревьев с бутстрэпом и без ограничения глубины. Сохраняем настройки sklearn по числу признаков на узел: `sqrt` для классификации и все признаки для регрессии. Меньшее, чем по умолчанию, число деревьев позволяет быстро повторять учебный эксперимент. Масштабирование не нужно. Для температуры отдельно показываем ошибку прогноза по текущему значению.

In [2]:
def classification_metrics(y, prediction):
    return {"f1_macro": f1_score(y, prediction, average="macro"), "accuracy": accuracy_score(y, prediction)}


def regression_metrics(y, prediction):
    return {"mae": mean_absolute_error(y, prediction), "rmse": root_mean_squared_error(y, prediction)}


clf = RandomForestClassifier(n_estimators=25, max_features="sqrt", random_state=RANDOM_STATE)
reg = RandomForestRegressor(n_estimators=25, max_features=1.0, random_state=RANDOM_STATE)
clf.fit(Xc_train, yc_train)
reg.fit(Xr_train, yr_train)
classification_scores = {"sklearn_base": classification_metrics(yc_test, clf.predict(Xc_test))}
regression_scores = {"sklearn_base": regression_metrics(yr_test, reg.predict(Xr_test))}
persistence = regression_metrics(yr_test, Xr_test[DINING_TEMPERATURE])

display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict({**regression_scores, "текущая температура": persistence}, orient="index"))

,f1_macro,accuracy
sklearn_base,0.944213,0.946492


,mae,rmse
sklearn_base,0.302118,0.703750
текущая температура,0.116360,0.133181


## 3. Улучшение бейзлайна

### 3a. Гипотезы

1. Рост числа деревьев с 25 до 60 может уменьшить разброс прогнозов ансамбля.
2. Половина признаков на узел может дать классификатору больше вариантов разбиения, а регрессору — более разнообразные деревья по сравнению с перебором всех признаков.

### 3b. Проверка гипотез

Для каждой задачи сравниваем четыре комбинации на обучающей части. Для классификации используем стратифицированные фолды и F1-macro, для регрессии — временные фолды с зазором и MAE.

In [3]:
clf_grid = {"n_estimators": [25, 60], "max_features": ["sqrt", 0.5]}
reg_grid = {"n_estimators": [25, 60], "max_features": [1.0, 0.5]}
clf_search = GridSearchCV(clf, clf_grid, scoring="f1_macro", cv=spambase_cv()).fit(Xc_train, yc_train)
reg_search = GridSearchCV(reg, reg_grid, scoring="neg_mean_absolute_error", cv=sml2010_cv()).fit(Xr_train, yr_train)


def cv_table(search, metric, sign=1):
    results = search.cv_results_
    return pd.DataFrame({
        "n_estimators": results["param_n_estimators"],
        "max_features": [str(value) for value in results["param_max_features"]],
        metric: sign * results["mean_test_score"],
    }).sort_values(metric, ascending=(sign == -1))


display(cv_table(clf_search, "CV F1-macro"))
display(cv_table(reg_search, "CV MAE, °C", sign=-1))

,n_estimators,max_features,CV F1-macro
1,60,sqrt,0.946330
0,25,sqrt,0.941716
3,60,0.5,0.934594
2,25,0.5,0.933032


,n_estimators,max_features,"CV MAE, °C"
0,25,1.0,0.128038
1,60,1.0,0.129384
2,25,0.5,0.188548
3,60,0.5,0.200390


### 3c. Улучшенный бейзлайн

Берём настройки с лучшей средней CV-метрикой. Выбранные модели уже переобучены на всей обучающей части. Оцениваем их на фиксированном тесте и сравниваем с пунктом 2.

In [4]:
print("Spambase:", clf_search.best_params_)
print("SML2010:", reg_search.best_params_)
classification_scores["sklearn_improved"] = classification_metrics(yc_test, clf_search.predict(Xc_test))
regression_scores["sklearn_improved"] = regression_metrics(yr_test, reg_search.predict(Xr_test))
display(pd.DataFrame.from_dict(classification_scores, orient="index"))
display(pd.DataFrame.from_dict(regression_scores, orient="index"))

Spambase: {'max_features': 'sqrt', 'n_estimators': 60}
SML2010: {'max_features': 1.0, 'n_estimators': 25}


,f1_macro,accuracy
sklearn_base,0.944213,0.946492
sklearn_improved,0.952672,0.954816


,mae,rmse
sklearn_base,0.302118,0.70375
sklearn_improved,0.302118,0.70375


**Выводы.** Для Spambase лучшим по CV оказался лес из 60 деревьев с `max_features='sqrt'`: F1-macro на тесте вырос с 0,9442 до 0,9527. Половина признаков на узел оказалась хуже `sqrt` при обоих размерах леса. Для SML2010 ни один вариант не обошёл базовый по CV MAE: оставили 25 деревьев и все признаки. Тестовая ошибка MAE 0,3021 °C выше, чем у прогноза по текущей температуре (0,1164 °C).

## 4. Имплементация алгоритма машинного обучения

### 4.1. Базовая реализация

Переиспользуем [ядро дерева из ЛР3](tree.py). Для каждого дерева выбираем столько же строк, сколько в обучающей выборке, но с возвращением. Дерево при каждом разбиении заново выбирает случайное подмножество признаков, если `max_features` меньше полного числа признаков. Бутстрэп и выбор признаков воспроизводимы благодаря фиксированному seed.

В регрессии усредняем прогнозы деревьев. В бинарной классификации усредняем доли класса `1` в листьях и выбираем спам, если результат больше 0,5. Это не то же самое, что голосование жёстких меток деревьев.

In [5]:
class RandomForestNumPy:
    def __init__(self, n_estimators=25, max_features="sqrt", classification=True, random_state=42):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.classification = classification
        self.random_state = random_state

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        rng = np.random.default_rng(self.random_state)
        self.trees = []
        for _ in range(self.n_estimators):
            sample = rng.integers(0, len(y), size=len(y))
            tree = DecisionTreeNumPy(
                classification=self.classification, max_features=self.max_features,
                random_state=int(rng.integers(0, 2**32)),
            )
            self.trees.append(tree.fit(X[sample], y[sample]))
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        mean = np.mean([tree.predict_leaf(X) for tree in self.trees], axis=0)
        return (mean > 0.5).astype(int) if self.classification else mean

На простом примере проверяем воспроизводимость деревьев и меток, а на константной регрессионной цели — усреднение. Затем обучаем собственные леса с настройками из пункта 2 и сравниваем на том же тесте.

In [6]:
X_toy = np.array([[0, 10], [1, 3], [2, 8], [3, 1], [4, 6], [5, 5]])
y_toy = [0, 0, 0, 1, 1, 1]
toy1 = RandomForestNumPy(n_estimators=9).fit(X_toy, y_toy)
toy2 = RandomForestNumPy(n_estimators=9).fit(X_toy, y_toy)
assert len(toy1.trees) == 9 and [t.tree for t in toy1.trees] == [t.tree for t in toy2.trees]
np.testing.assert_array_equal(toy1.predict(X_toy), toy2.predict(X_toy))
toy_reg = RandomForestNumPy(n_estimators=5, classification=False).fit(X_toy, np.full(len(X_toy), 2.5))
np.testing.assert_allclose(toy_reg.predict(X_toy), 2.5)

clf_own = RandomForestNumPy(random_state=RANDOM_STATE).fit(Xc_train, yc_train)
reg_own = RandomForestNumPy(classification=False, max_features=1.0, random_state=RANDOM_STATE).fit(Xr_train, yr_train)
classification_scores["custom_base"] = classification_metrics(yc_test, clf_own.predict(Xc_test))
regression_scores["custom_base"] = regression_metrics(yr_test, reg_own.predict(Xr_test))
base_variants = ["sklearn_base", "custom_base"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[base_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[base_variants])

,f1_macro,accuracy
sklearn_base,0.944213,0.946492
custom_base,0.946584,0.948870


,mae,rmse
sklearn_base,0.302118,0.703750
custom_base,0.285158,0.670294


**Сравнение с пунктом 2.** Базовый NumPy-лес получил F1-macro 0,9466 против 0,9442 у sklearn и MAE 0,2852 °C против 0,3021 °C. Один seed делает каждый эксперимент воспроизводимым, но не синхронизирует бутстрэп и выбор порогов у реализаций. Кроме того, наше дерево не проверяет дополнительные признаки, если ни один из выбранных не даёт допустимого разбиения.

### 4.2. Улучшенная реализация

Переносим число деревьев и число признаков на узел из пункта 3c. Повторно гиперпараметры не подбираем. Сравниваем с улучшенными моделями sklearn на том же тесте.

In [7]:
clf_own_improved = RandomForestNumPy(random_state=RANDOM_STATE, **clf_search.best_params_).fit(Xc_train, yc_train)
reg_own_improved = RandomForestNumPy(
    classification=False, random_state=RANDOM_STATE, **reg_search.best_params_,
).fit(Xr_train, yr_train)
classification_scores["custom_improved"] = classification_metrics(yc_test, clf_own_improved.predict(Xc_test))
regression_scores["custom_improved"] = regression_metrics(yr_test, reg_own_improved.predict(Xr_test))
improved_variants = ["sklearn_improved", "custom_improved"]
display(pd.DataFrame.from_dict(classification_scores, orient="index").loc[improved_variants])
display(pd.DataFrame.from_dict(regression_scores, orient="index").loc[improved_variants])

,f1_macro,accuracy
sklearn_improved,0.952672,0.954816
custom_improved,0.952771,0.954816


,mae,rmse
sklearn_improved,0.302118,0.703750
custom_improved,0.285158,0.670294


**Сравнение с пунктом 3.** В классификации 60 деревьев подняли F1-macro собственного леса с 0,9466 до 0,9528 (sklearn: 0,9527). В регрессии выбранные настройки совпали с базовыми: MAE собственного леса осталась 0,2852 °C. Ни один лес не превзошёл прогноз по текущей температуре на этом тесте.

Сохраняем четыре варианта на задачу в том же формате, что в ЛР1–3. Прогноз по текущей температуре — ориентир, в CSV не включаем.

In [8]:
variants = ["sklearn_base", "sklearn_improved", "custom_base", "custom_improved"]
rows = []
for task, scores in [("classification", classification_scores), ("regression", regression_scores)]:
    for variant in variants:
        rows.append({"lab": 4, "task": task, "algorithm": "RandomForest", "variant": variant, **scores[variant]})

results = pd.DataFrame(rows)
results.to_csv(root / "results" / "lab04.csv", index=False, float_format="%.8f")
for task in results["task"].unique():
    display(results.loc[results["task"] == task].dropna(axis=1, how="all").reset_index(drop=True))

,lab,task,algorithm,variant,f1_macro,accuracy
0,4,classification,RandomForest,sklearn_base,0.944213,0.946492
1,4,classification,RandomForest,sklearn_improved,0.952672,0.954816
2,4,classification,RandomForest,custom_base,0.946584,0.948870
3,4,classification,RandomForest,custom_improved,0.952771,0.954816


,lab,task,algorithm,variant,mae,rmse
0,4,regression,RandomForest,sklearn_base,0.302118,0.703750
1,4,regression,RandomForest,sklearn_improved,0.302118,0.703750
2,4,regression,RandomForest,custom_base,0.285158,0.670294
3,4,regression,RandomForest,custom_improved,0.285158,0.670294
